# Short notebook 1 — TDT4173 course project, Tokke–Vinje unit commitment

| | |
|---|---|
| **Kaggle team name** | `[25] Nah, i'd win`|
| **Members** | `Francesco Binni — student ID` |
| **Members** | `Cece — student ID` |
| **Members** | `Cece — student ID` |

This notebook reproduces a weighted average of
XGBoost (20%), LightGBM (30%), a sequence neural network (25%) and a "refiner" sequence network (25%). The refiner is the same sequence network with one extra input, the probability that XGBoost predicts for each unit and hour, which the network learns to correct by looking at the whole week. 
During training, the network is asked to predict the true commitment status (0 = off, 1 = on). Across thousands of training weeks it observes both the XGBoost prediction and the actual outcome, and from these it learns two things:

- where XGBoost is reliable, the best estimate is to retain its probability;
- where XGBoost tends to make systematic errors, the probability should be adjusted accordingly.

For the training cases this input is an out-of-fold prediction: each year is predicted by an XGBoost trained only on the other years, so the refiner learns from predictions with realistic errors rather than from predictions that have already seen the labels. 


The exploratory analysis, the feature analysis, all the models we tried, the model interpretation and **how
every hyper-parameter value written below was obtained** are in the PDF report.

**How to run**
1. Install the packages listed in `requirements.txt` (Python 3.12).

2. Place the competition data in the same folder as this notebook, with this layout:
   ```
   ├── Short notebook 1.ipynb
   ├── prediction_mapping.csv
   ├── sample_submission.csv
   └── data/
       ├── kernel/
       └── extended/
   ```
   No internet access and no other file is needed.

3. Run all cells. The prediction is written to `submission.csv` next to the notebook (725 rows × 2353 columns).

**Hardware and running time**

- Developed and run on an **NVIDIA GeForce RTX 5060 Laptop GPU (8 GB)** with CUDA 13.0 and PyTorch 2.14.1+cu130
  (the CUDA build of PyTorch is required to use the GPU). With this GPU the notebook needs about **1.1 h**.

- **Without a GPU** it runs on the CPU automatically: about **8.5 h on 4 CPU threads**, measured per step. If the notebook is run on CPU, to stay within the 12 h limit, the two neural networks (the sequence network and the refiner) are then averaged over 2 seeds instead of 5.

- If XGBoost cannot run on the GPU (e.g. less than 8 GB of GPU memory) it switches to the CPU by itself.

- Peak memory is about 18 GB of RAM.

**Reproducibility.** All random seeds are fixed. Small differences can come from GPU non-determinism and, on
CPU, from the smaller number of network seeds.


## 1. Setup
Imports the libraries and locates the competition files by name in this folder (or its parent), so no path
has to be edited. It also selects the GPU when available, otherwise the CPU, uses all CPU threads, and sets
the number of seeds of the two neural networks (5 on GPU, 2 on CPU). The output file is `submission.csv`.


In [ ]:
import os, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import torch
import xgboost as xgb
import lightgbm as lgb
warnings.filterwarnings("ignore")

def find_file(name, max_depth=4):
    # locate an original competition file below this folder (or its parent), by file name
    for root in (Path.cwd(), Path.cwd().parent):
        for depth in range(max_depth + 1):
            hits = sorted(p for p in root.glob("/".join(["*"] * depth + [name]))
                          if not any(part.startswith(".") for part in p.relative_to(root).parts))
            if hits:
                return hits[0]
    raise FileNotFoundError(f"{name} not found: place the competition data next to this notebook")


KERNEL = find_file("Unit_commitment_decisions.csv").parent          # data/kernel
EXTENDED = find_file("Tokke_Vinje_topology.yaml").parent             # data/extended
MAPPING_CSV = find_file("prediction_mapping.csv")
SAMPLE_CSV = find_file("sample_submission.csv")
OUT_CSV = Path("submission.csv")
N_THREADS = os.cpu_count()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(N_THREADS)
NN_SEEDS = 5 if DEVICE == "cuda" else 2  
T0 = time.time()
print("data:", KERNEL.parent.resolve(), "| device:", DEVICE, "| threads:", N_THREADS)


## 2. Data
Reads the original competition files: reservoir volumes, water values, inflows, day-ahead prices, the
environmental constraints (minimum flow and minimum volume), the topology of the system, the training labels
(`Unit_commitment_decisions.csv`) and the list of test cases (`prediction_mapping.csv`). It then builds two
structures:
- `cases`: a table with one row per case (a one-week SHOP run starting on a given day), with its `Run No`,
  start date and a train/test flag. It lists the 2,922 training cases (2015–2022) followed by the 725 test
  cases (2023–2024), in submission order.
- `labels`: a 3D array of shape (2,922 cases, 14 units, 168 hours) with the commitment status (1 = on,
  0 = off) of the training cases. In the CSV each case is a single row of 14 × 168 columns; here it is
  reshaped into one 14 × 168 grid per case.

In the feature engineering step, `cases` gives the start date of each week, which is used to collect that
week's prices, inflows, volumes and water values and to compute one row of features for every
(case, unit, hour). `labels`, flattened in the same order, gives the 0/1 target of each training row,
which is what the models learn to predict.



In [ ]:
H = 168
read = lambda p: pd.read_csv(p, parse_dates=["date"]).set_index("date")
volume = read(KERNEL / "Historical_volume_2015_2024.csv")
wv = read(KERNEL / "Synthetic_water_value_2015_2024.csv")
inflow = read(KERNEL / "Historical_inflow_1958_2025.csv")
price_s = read(KERNEL / "Historical_day_ahead_price_2015_2025.csv").iloc[:, 0]
min_flow = read(EXTENDED / "Constraint_min_flow.csv")
min_vol = read(EXTENDED / "Constraint_min_volume.csv")
uc = pd.read_csv(KERNEL / "Unit_commitment_decisions.csv")
mapping = pd.read_csv(MAPPING_CSV)
topo = yaml.safe_load(open(EXTENDED / "Tokke_Vinje_topology.yaml"))["model"]

UNITS = ["Hogga_G1", "Tokke_G1", "Tokke_G2", "Tokke_G3", "Tokke_G4", "Lio_G1", "Byrte_G1",
         "Vinje_G1", "Vinje_G2", "Vinje_G3", "Haukeli_G1", "Songa_G1", "Kjela_G1", "Vesle Kjela_G1"]
U = len(UNITS)
PLANT = [u.rsplit("_", 1)[0] for u in UNITS]
LABEL_COLS = [f"result_committed_{u}_t{t}" for u in UNITS for t in range(H)]

# --- Structure 1: `cases`
cases = pd.concat([uc[["Run No", "starttime"]].assign(is_train=True),
                   mapping[["Run No", "starttime"]].assign(is_train=False)], ignore_index=True)
cases["start"] = pd.to_datetime(cases["starttime"]).dt.tz_localize("UTC")
n_cases, n_train = len(cases), int(cases.is_train.sum())

# --- Structure 2: `labels`
labels = uc[LABEL_COLS].to_numpy(np.int8).reshape(n_train, U, H)
print(f"{n_train} train cases, {n_cases - n_train} test cases, ON rate {labels.mean():.3f}")


## 3. Topology structure
The `connections` section of `Tokke_Vinje_topology.yaml` is used to build, by hand, data structures that
summarise how reservoirs and plants are linked: the reservoir feeding each plant (`UP`), the reservoir
receiving its discharge (`DOWN`), reservoirs joined by tunnels at the same level that act as a single storage
(`UP_GROUP`), the reservoirs two levels upstream (`UP2`) and the whole catchment draining into each intake
(`CATCHMENT`). The input data are given per reservoir while the targets are per unit, so these structures
tell the feature engineering step which reservoirs to look at for each unit. The same file also provides
the maximum volume of each reservoir and the characteristics of each unit (maximum power, start-up cost and
maximum discharge, taken as the largest flow of its turbine curves).


In [ ]:
UP = {"Hogga": ["Bandak"], "Tokke": ["Vinjevatn"], "Lio": ["Byrtevatn"], "Byrte": ["Botnedalsvatn"],
      "Vinje": ["Vaamarvatn"], "Haukeli": ["Vatjern"], "Songa": ["Songavatn", "Bitdalsvatn"],
      "Kjela": ["Foersvatn"], "Vesle Kjela": ["Kjelavatn"]}
DOWN = {"Hogga": [], "Tokke": ["Bandak"], "Lio": ["Bandak"], "Byrte": ["Byrtevatn"],
        "Vinje": ["Vinjevatn"], "Haukeli": ["Vinjevatn"], "Songa": ["Totak"], "Kjela": ["Hyljelihyl"],
        "Vesle Kjela": ["Foersvatn"]}
UP_GROUP = {"Hogga": ["Bandak"], "Tokke": ["Vinjevatn"], "Lio": ["Byrtevatn"],
            "Byrte": ["Botnedalsvatn"], "Vinje": ["Vaamarvatn", "Totak", "Venemo"],
            "Haukeli": ["Vatjern", "Langeidvatn"], "Songa": ["Songavatn", "Bitdalsvatn"],
            "Kjela": ["Foersvatn", "Bordalsvatn"], "Vesle Kjela": ["Kjelavatn", "Staavatn"]}
UP2 = {"Hogga": ["Vinjevatn", "Byrtevatn"], "Tokke": ["Vaamarvatn", "Totak", "Langeidvatn"],
       "Lio": ["Botnedalsvatn"], "Byrte": [], "Vinje": ["Songavatn", "Bitdalsvatn"],
       "Haukeli": [], "Songa": [], "Kjela": ["Langesae"], "Vesle Kjela": []}
_KJELA = ["Foersvatn", "Bordalsvatn", "Langesae", "Kjelavatn", "Staavatn"]
_VINJE = ["Vaamarvatn", "Totak", "Venemo", "Songavatn", "Bitdalsvatn", "Hyljelihyl"] + _KJELA
CATCHMENT = {"Hogga": None, "Tokke": ["Vinjevatn", "Vatjern", "Langeidvatn"] + _VINJE,
             "Lio": ["Byrtevatn", "Botnedalsvatn"], "Byrte": ["Botnedalsvatn"], "Vinje": _VINJE,
             "Haukeli": ["Vatjern", "Langeidvatn"], "Songa": ["Songavatn", "Bitdalsvatn"],
             "Kjela": _KJELA, "Vesle Kjela": ["Kjelavatn", "Staavatn"]}
MAX_VOL = {k: v["max_vol"] for k, v in topo["reservoir"].items()}
GEN = topo["generator"]
QMAX = {u: max(max(c["x"]) for c in GEN[u]["turb_eff_curves"]) for u in UNITS}
print("max discharge (m3/s):", QMAX)


## 4. Feature engineering
The models always answer the same question: *"in this week, is this unit ON at this hour?"*. Each row of the
feature table therefore corresponds to one of these questions, to one combination of:
- **case**: which week (e.g. the one starting on 1 January 2015);
- **unit**: which generator (e.g. Tokke_G1);
- **hour**: which of the 168 hours of the week (e.g. hour 40).

| case | unit | hour | price | margin m | reservoir volume | … (220 features) | target |
|---|---|---:|---:|---:|---:|---|---:|
| 2015-01-01 | Hogga_G1 | 0 | 30.1 | 2.4 | … | … | 1 |
| 2015-01-01 | Hogga_G1 | 1 | 29.8 | 2.1 | … | … | 1 |
| … | … | … | … | … | … | … | … |
| 2015-01-01 | Hogga_G1 | 167 | … | … | … | … | 0 |
| 2015-01-01 | Tokke_G1 | 0 | 30.1 | 5.7 | … | … | 1 |
| … | … | … | … | … | … | … | … |
| 2015-01-02 | Hogga_G1 | 0 | … | … | … | … | … |

*(values are illustrative)*

- Each case produces 14 units × 168 hours = 2,352 rows; in total 3,647 cases × 2,352 ≈ **8.6 million rows**.
- The **columns** are the 220 features, which will be summarized below.
- The table contains both the 2,922 training cases (≈ 6.9 million rows) and, after them, the 725 test cases
  (≈ 1.7 million rows), with features computed in exactly the same way. Only the training rows have a 0/1
  target, kept in a separate array (`labels` flattened in the same order); the test rows are the ones the
  models must predict.


XGBoost and LightGBM look at one row at a time and learn to map its 220 features to the target.

### Main feature groups
The groups below follow the comments inside `build_features`. The full analysis of which groups are useful
 is in the PDF report.

| Group (comment in the code) | N of features | What it describes | Examples |
|---|---:|---|---|
| Raw state of every reservoir | 111 | Volume, water value at the start and at d+7, weekly inflow of each of the 17 reservoirs; price minus each reservoir's d+7 water value; weekly mean of the minimum-flow and minimum-volume constraints | `V0_*`, `W0_*`, `W7_*`, `dW_*`, `Iwk_*`, `pmW7_*`, `mf_*`, `mv_*` |
| Price shape of the week | 22 | Hourly price, its rank and z-score within the week, rolling means, weekly quantiles, calendar, hour and unit index | `price`, `p_rank`, `p_z`, `p_c24`, `p_q90`, `doy` |
| Per-unit economics from the topology | 40 | **The most important features of the model.** Margin `m` = price − d+7 water value of the unit's upstream reservoir: the unit tends to be on when `m` > 0. Variants of this margin are 8 of the 10 features with the highest XGBoost gain. The group also includes upstream/downstream water values and the temporal context of `m` (rolling stats, share of hours with `m` > 0, distance to the next sign change) | `m`, `m_cmean7`, `m_frac_week`, `m_dnext`, `u_netW7` |
| Effective storage, spill risk, unit data | 20 | Fill level of tunnel-connected reservoirs, risk of spilling, hourly inflow, unit power and start-up cost | `g_fill`, `g_spill_risk`, `g_inflow_h`, `gen_pmax`, `gen_start_per_mw` |
| Hydro budget in hours | 9 | Water available or arriving, expressed as hours of full production of the plant/unit | `hb_storage_local_h`, `hb_inflow_catch_h`, `hb_spill_local_h` |
| Hourly environmental constraints | 18 | Minimum flow and minimum volume per hour, the minimum-flow deficit on the river Lio discharges into, normalised margins | `mfh_*`, `rvest_deficit_h`, `mvmargin_*`, `m_over_pstd` |
| **Total** | **220** | | |







In [ ]:


# For each unit, combines the values of the reservoirs linked to its plant in a topology structure
# (UP, DOWN, UP_GROUP, ...), e.g. summing their volumes or averaging their water values.
# Plants with no linked reservoir get 0.
def _per_unit(mapping, values, agg):
    """values: reservoir -> array (n, ...). Returns (n, U, ...) aggregated over each unit's reservoirs."""
    out = []
    for p in PLANT:
        rs = mapping[p]
        out.append(agg(np.stack([values[r] for r in rs]), axis=0) if rs
                   else np.zeros_like(next(iter(values.values()))))
    return np.stack(out, axis=1)


# Centred rolling statistic (mean, min or max) over a window of w hours, computed along the week
# for every case and unit.
def _rolling(a, w, fn):
    n, u, h = a.shape
    s = pd.DataFrame(a.reshape(-1, h).T).rolling(w, center=True, min_periods=1)
    return getattr(s, fn)().to_numpy().T.reshape(n, u, h)


# For a true/false series along the week (e.g. "margin > 0"): number of hours since the last change
# (dprev) and until the next change (dnext).
def _run_lengths(above):
    dprev = np.zeros(above.shape, np.float32); dnext = np.zeros(above.shape, np.float32)
    for t in range(1, H):
        dprev[..., t] = np.where(above[..., t] == above[..., t - 1], dprev[..., t - 1] + 1, 0)
    for t in range(H - 2, -1, -1):
        dnext[..., t] = np.where(above[..., t] == above[..., t + 1], dnext[..., t + 1] + 1, 0)
    return dprev, dnext


# Adds a margin and its temporal context to the feature dictionary f: rolling means/min/max, share of
# hours with a positive margin in the week and in the day, distance to the sign changes, weekly
# quantiles and sums. Used for the margin m and, in a shorter version, for the group margin mg.
def _margin_context(m, prefix, f, full=True):
    """Temporal context of a per-unit margin m (n, U, H): rolling stats, share of hours above zero,
    distance to the next/previous sign change, weekly quantiles."""
    f[prefix] = m
    for w in ((3, 7, 13, 25) if full else (7, 25)):
        f[f"{prefix}_cmean{w}"] = _rolling(m, w, "mean")
        if full:
            f[f"{prefix}_cmin{w}"] = _rolling(m, w, "min"); f[f"{prefix}_cmax{w}"] = _rolling(m, w, "max")
    above = m > 0
    f[f"{prefix}_above"] = above
    f[f"{prefix}_frac_week"] = above.mean(-1, keepdims=True)
    f[f"{prefix}_frac_day"] = np.repeat(above.reshape(*above.shape[:2], 7, 24).mean(-1), 24, -1)
    if full:
        dprev, dnext = _run_lengths(above)
        f[f"{prefix}_dprev"], f[f"{prefix}_dnext"], f[f"{prefix}_runlen"] = dprev, dnext, dprev + dnext + 1
        for q in (20, 50, 80):
            f[f"{prefix}_wq{q}"] = np.percentile(m, q, axis=-1, keepdims=True)
        f[f"{prefix}_pos_sum"] = np.clip(m, 0, None).sum(-1, keepdims=True)
        f[f"{prefix}_neg_sum"] = np.clip(m, None, 0).sum(-1, keepdims=True)


# Builds the 220 features of the given cases, group by group (see the table above), and returns them
# as one float32 matrix with one row per (case, unit, hour), together with the feature names.
def build_features(cases):
    st = pd.DatetimeIndex(cases["start"]); n = len(st)
    hours = pd.DatetimeIndex((st.values[:, None] + np.arange(H) * np.timedelta64(1, "h")).ravel(), tz="UTC")
    hourly = lambda s: s.reindex(hours).to_numpy().reshape(n, H)
    res = list(volume.columns)
    price = hourly(price_s); assert not np.isnan(price).any()
    f = {}
    C = lambda a: np.asarray(a, np.float32).reshape(n, 1, 1)   # case-level
    T = lambda a: np.asarray(a, np.float32).reshape(n, 1, H)   # case x hour


    # RAW STATE OF EVERY RESERVOIR
    V0 = {r: volume[r].reindex(st).to_numpy() for r in res}
    W0 = {r: wv[r].reindex(st).to_numpy() for r in res}
    W7 = {r: wv[r].reindex(st + pd.Timedelta(days=7)).to_numpy() for r in res}   # end-of-horizon value
    inflow_h = {r: hourly(inflow[r]) for r in res}
    for r in res:
        f[f"V0_{r}"] = C(V0[r]); f[f"W0_{r}"] = C(W0[r]); f[f"W7_{r}"] = C(W7[r])
        f[f"dW_{r}"] = C(W7[r] - W0[r]); f[f"Iwk_{r}"] = C(inflow_h[r].mean(1))
        f[f"pmW7_{r}"] = T(price - W7[r][:, None])
    for c in min_flow.columns:
        f[f"mf_{c}"] = C(hourly(min_flow[c]).mean(1))
    for c in min_vol.columns:
        f[f"mv_{c}"] = C(hourly(min_vol[c]).mean(1))



    # PRICE SHAPE OF THE WEEK
    f["p_mean"] = C(price.mean(1)); f["p_std"] = C(price.std(1))
    for q in (0, 10, 25, 50, 75, 90, 100):
        f[f"p_q{q}"] = C(np.percentile(price, q, axis=1))
    f["doy"] = C(st.dayofyear); f["dow0"] = C(st.dayofweek)
    f["price"] = T(price)
    f["p_rank"] = T(price.argsort(1).argsort(1) / (H - 1))
    f["p_minus_mean"] = T(price - price.mean(1, keepdims=True))
    f["p_z"] = T((price - price.mean(1, keepdims=True)) / (price.std(1, keepdims=True) + 1e-3))
    for w in (3, 6, 12, 24):
        f[f"p_c{w}"] = T(pd.DataFrame(price.T).rolling(w, center=True, min_periods=1).mean().to_numpy().T)
    f["p_day_rank"] = T(price.reshape(n, 7, 24).argsort(-1).argsort(-1).reshape(n, H) / 23)
    f["t"] = np.arange(H, dtype=np.float32).reshape(1, 1, H)
    f["unit"] = np.arange(U, dtype=np.float32).reshape(1, U, 1)


    #PER-UNIT ECONOMICS FROM THE TOPOLOGY
    for name, d in (("W7", W7), ("W0", W0), ("V0", V0), ("Iwk", {r: inflow_h[r].mean(1) for r in res})):
        agg = np.mean if name[0] == "W" else np.sum
        f[f"u_up_{name}"] = _per_unit(UP, d, agg)[:, :, None]
        f[f"u_dn_{name}"] = _per_unit(DOWN, d, agg)[:, :, None]
    up_w7 = f["u_up_W7"]
    f["u_netW7"] = up_w7 - f["u_dn_W7"]; f["u_netW0"] = f["u_up_W0"] - f["u_dn_W0"]
    P = price[:, None, :]
    f["u_p_m_net"] = P - f["u_netW7"]
    for k in ("p_mean", "p_q75", "p_q25"):
        f[f"u_{k}_m_net"] = f[k] - f["u_netW7"]
    f["u_ratio"] = P / (np.abs(up_w7) + 1)
    f["u_p_m_upW0"] = P - f["u_up_W0"]
    _margin_context(P - up_w7, "m", f)   # the key feature: price - WV_d+7(upstream)



    #EFFECTIVE STORAGE, SPILL RISK AND INFLOW
    Vg = _per_unit(UP_GROUP, V0, np.sum)
    capg = _per_unit(UP_GROUP, {r: np.full(n, MAX_VOL[r]) for r in res}, np.sum)
    inflow_mcm_wk = {r: inflow_h[r].sum(1) * 3600 / 1e6 for r in res}
    Ig = _per_unit(UP_GROUP, inflow_mcm_wk, np.sum)
    f["g_fill"] = (Vg / capg)[:, :, None]
    f["g_inflow_over_cap"] = (Ig / capg)[:, :, None]
    f["g_spill_risk"] = ((Vg + Ig - capg) / capg)[:, :, None]
    f["g_headroom_days"] = ((capg - Vg) / (Ig / 7 + 1e-3))[:, :, None]
    f["g_W7"] = (_per_unit(UP_GROUP, {r: W7[r] * MAX_VOL[r] for r in res}, np.sum) / capg)[:, :, None]
    _margin_context(P - f["g_W7"], "mg", f, full=False)
    has_up2 = np.array([len(UP2[p]) > 0 for p in PLANT], np.float32).reshape(1, U, 1)
    up2 = _per_unit(UP2, W7, np.mean)[:, :, None]
    f["u_up2_W7"] = np.where(has_up2 > 0, up2, np.nan)
    f["m2"] = np.where(has_up2 > 0, P - up2, np.nan)
    Ih = _per_unit(UP_GROUP, inflow_h, np.sum)
    f["g_inflow_h"] = Ih
    f["g_inflow_c24"] = _rolling(Ih, 25, "mean")
    f["gen_pmax"] = np.array([GEN[u]["p_max"] for u in UNITS], np.float32).reshape(1, U, 1)
    f["gen_start_per_mw"] = np.array([GEN[u]["startcost_const"] / GEN[u]["p_max"] for u in UNITS],
                                     np.float32).reshape(1, U, 1)
    f["plant_n_units"] = np.array([PLANT.count(p) for p in PLANT], np.float32).reshape(1, U, 1)
    f["unit_idx_in_plant"] = np.array([int(u.rsplit("_G", 1)[1]) for u in UNITS], np.float32).reshape(1, U, 1)
    f["m_minus_startcost"] = f["m_cmean7"] - f["gen_start_per_mw"] / 7



    #HYDRO BUDGET AND ENVIRONMENTAL CONSTRAINTS
    catch = {p: (res if c is None else c) for p, c in CATCHMENT.items()}
    q_plant = np.array([sum(QMAX[u] for u in UNITS if u.startswith(p + "_")) for p in PLANT])
    q_unit = np.array([QMAX[u] for u in UNITS])
    to_h = lambda mcm, q: (mcm * 1e6 / (q[None, :] * 3600))[:, :, None]
    I_c = _per_unit(catch, inflow_mcm_wk, np.sum)
    V_c = _per_unit(catch, V0, np.sum)
    cap_c = _per_unit(catch, {r: np.full(n, MAX_VOL[r]) for r in res}, np.sum)
    f["hb_inflow_local_h"] = to_h(Ig, q_plant)
    f["hb_inflow_catch_h"] = to_h(I_c, q_plant)
    f["hb_storage_local_h"] = to_h(Vg, q_plant)
    f["hb_headroom_local_h"] = to_h(capg - Vg, q_plant)
    f["hb_spill_local_h"] = f["hb_inflow_local_h"] - f["hb_headroom_local_h"]
    f["hb_spill_catch_h"] = to_h(V_c + I_c - cap_c, q_plant)
    f["hb_inflow_unit_h"] = to_h(Ig, q_unit)
    f["catch_fill"] = (V_c / cap_c)[:, :, None]
    fwd24 = pd.DataFrame(Ih.reshape(-1, H).T[::-1]).rolling(24, min_periods=1).mean()
    f["hb_inflow_next24"] = fwd24.to_numpy()[::-1].T.reshape(n, U, H) / q_plant[None, :, None]



    #HOURLY ENVIRONMENTAL CONSTRAINTS
    mf = {c: hourly(min_flow[c]) for c in min_flow.columns}
    for c, v in mf.items():
        f[f"mfh_{c}"] = T(v)
    deficit = mf["r_Vest_Vassdraget"] - hourly(inflow["r_Vest_Vassdraget"])   # river Lio discharges into
    f["rvest_deficit_h"] = T(deficit)
    f["rvest_deficit_over_lio"] = T(np.clip(deficit, 0, None) / QMAX["Lio_G1"])
    f["rvest_deficit_week"] = C(np.clip(deficit, 0, None).mean(1))
    for c in min_vol.columns:
        mvh = hourly(min_vol[c])
        f[f"mvmargin_{c}"] = C(V0[c] - mvh.max(1)); f[f"mvh_{c}"] = T(mvh)
    f["m_over_pstd"] = f["m"] / (price.std(1)[:, None, None] + 1)
    f["m_over_W7"] = f["m"] / (np.abs(up_w7) + 1)
    f["m_c7_over_pstd"] = f["m_cmean7"] / (price.std(1)[:, None, None] + 1)

    #FLATTEN INTO ONE FLOAT32 MATRIX, ROWS ORDERED CASE -> UNIT -> HOUR (= SUBMISSION ORDER)
    names = list(f)
    X = np.empty((n * U * H, len(names)), np.float32)
    for j, k in enumerate(names):
        X[:, j] = np.broadcast_to(np.asarray(f.pop(k), np.float32), (n, U, H)).ravel()
    return X, names


Runs the feature construction on all cases (training and test). `X` has one row per (case, unit, hour)
and 220 columns; the training rows come first, so `X[:n_tr_rows]` is the training part and
`X[n_tr_rows:]` the test part. `y_tr` is the 0/1 target of the training rows, in the same order.


In [ ]:
t = time.time()
X, FEATURES = build_features(cases)
n_tr_rows = n_train * U * H                 # train rows come first: X[:n_tr_rows] is a view, no copy
y_tr = labels.ravel()
print(X.shape, f"{X.nbytes / 1e9:.1f} GB, {time.time() - t:.0f}s")


## 5. Gradient boosting (XGBoost)
The first model of the blend is an XGBoost classifier, trained on all the training rows (2015–2022) of the
feature table and used to predict the probability that each unit is on in each hour of the test cases.
XGBoost builds an ensemble of decision trees, each one correcting the errors of the previous ones.

**Hyper-parameters.** 

The values used here come from a small grid search done during development: a few combinations of depth, learning rate, min_child_weight, colsample_bytree and reg_lambda, each evaluated with the temporal cross-validation (folds 2020, 2021, 2022). 
Depth 10 with min_child_weight 50 and learning rate 0.03 gave the best mean AUC; the other values were kept at their initial setting. 
We did not run an exhaustive search for two reasons. First, the differences between the tested combinations were small (about 0.002 in mean AUC between the worst and the best), comparable to or smaller than the gains obtained from better features and from blending different models. Second, testing many combinations on the same three validation years increases the risk of selecting a setting that is best on those years by chance, i.e. of overfitting the validation. No tuning is done in this notebook.

The code trains on the GPU when available and switches to the CPU automatically if the GPU fails (e.g. not
enough GPU memory); `max_bin` is set to 128 so that training fits in 8 GB of GPU memory. The averaged test
prediction `p_xgb` is used in the final blend with weight 0.20.



In [ ]:
XGB_DEVICE = DEVICE


def train_xgb(params, dtrain, rounds):
    """Train on the GPU when available; fall back to the CPU if the GPU fails (e.g. not enough memory)."""
    global XGB_DEVICE
    try:
        return xgb.train({**params, "device": XGB_DEVICE}, dtrain, rounds)
    except xgb.core.XGBoostError as err:
        if XGB_DEVICE == "cpu":
            raise
        print("XGBoost on GPU failed, switching to CPU:", str(err).splitlines()[0][:120])
        XGB_DEVICE = "cpu"
        return xgb.train({**params, "device": "cpu"}, dtrain, rounds)


xgb_params = dict(objective="binary:logistic", tree_method="hist", device=DEVICE, max_depth=10,
                  learning_rate=0.03, subsample=0.8, colsample_bytree=0.6, min_child_weight=50,
                  reg_lambda=5, max_bin=128, nthread=N_THREADS)
t = time.time()
dtr = xgb.QuantileDMatrix(X[:n_tr_rows], y_tr, max_bin=128, nthread=N_THREADS)
dte = xgb.DMatrix(X[n_tr_rows:], nthread=N_THREADS)
p_xgb = []
for seed in range(3):
    booster = train_xgb({**xgb_params, "seed": seed}, dtr, 800)
    p_xgb.append(booster.predict(dte)); print(f"xgb seed {seed} done, {time.time() - t:.0f}s", flush=True)
p_xgb = np.mean(p_xgb, 0).reshape(-1, U, H)
del dtr, dte, booster


## 6. Out-of-fold XGBoost predictions (input of the refiner network)

The refiner has one extra input column.** The first sequence network receives, for every
(case, unit, hour), the 220 features. The refiner receives the same 220 features **plus one column**:
The probability that XGBoost assigns to that cell.

| case | unit | hour | 220 features | **column 221: XGBoost prediction** |
|---|---|---:|---|---:|
| … | … | … | … | 0.83 |

The refiner uses this column at two moments.**
- **Training**, on the training cases (2015–2022): the network sees the 221 columns and the true answer, and learns.
- **Prediction**, on the test cases (2023–2024): the network sees the 221 columns and outputs its probability.

At both moments column 221 must be filled with something.

**What column 221 is filled with.**

| Rows | Value of column 221 | From which XGBoost |
|---|---|---|
| Training 2015–2022 | `oof` | 8 out-of-fold XGBoost models: each year predicted by a model trained on the other years |
| Test 2023–2024 | `p_xgb` | the final XGBoost of Section 5, trained on all of 2015–2022 |

`p_xgb` is therefore the refiner's input on the test cases, and the out-of-fold `oof` prediction is its input on the
training cases.

**Why two different XGBoost models.** 

Column 221 must **mean the same thing** in training and in test:
*"the prediction of an XGBoost that has never seen this case"*.
- **Test:** the final XGBoost has never seen the 2023–2024 cases, so `p_xgb` is an honest prediction, with its
  normal errors.
- **Training, if the final XGBoost were used:** that model was trained on exactly the 2015–2022 cases, so it
  has already seen their answers. Its predictions would be "too good", e.g. 0.99 where the answer is 1 and
  0.01 where it is 0.
- **Training with out-of-fold predictions:** each year is predicted by a model that has not seen it, so the
  predictions contain realistic errors, as on the test cases.

If the refiner were trained with the first row, it would learn that column 221 is always right and simply copy
it; on the test cases, where the column is sometimes wrong, it could not notice. With out-of-fold predictions,
during training it sees a column that makes the same kind of errors it will make on the test cases, so it learns
**when to trust it and when to correct it**. In short, the two XGBoost models are different so that their output
is equally reliable in training and in test.

**How the out-of-fold predictions are computed.** 

The code repeats the following for each of the 8 training years:

```
Year to predict: 2015 → train XGBoost on 2016–2022       → predict 2015
Year to predict: 2016 → train XGBoost on 2015, 2017–2022 → predict 2016
...
Year to predict: 2022 → train XGBoost on 2015–2021       → predict 2022
```

**Purge.** 

Each case covers 7 days and a new case starts every day, so neighbouring cases overlap: when
predicting 2018, the case starting on 28 December 2017 already covers the first days of 2018. If it stayed in
the training data, the model would see the answers for hours that also appear in the 2018 cases to predict.
Therefore the cases starting within 7 days before or after the held-out year are also removed from training:

```
... 2017 ... | 7 days removed | 2018 (to predict) | 7 days removed | ... 2019 ...
   training                                                            training
```

**What is done with the result.** 

`oof` (training cases) and `p_xgb` (test cases) are joined into one array,
`channel`, with one probability for every (case, unit, hour) of all 3,647 cases. It is **not** added to the
feature table used by the tree models: in Section 9 it is converted to a logit, standardised with training
statistics only, and appended as column 221 for the refiner network only. Each of the 8 out-of-fold models is
trained with a single seed, to limit the running time.


In [ ]:
t = time.time()
train_start = cases.start[:n_train].reset_index(drop=True)
train_year = train_start.dt.year.to_numpy()
oof = np.zeros((n_train, U, H), np.float32)
for vy in np.unique(train_year):
    lo = pd.Timestamp(f"{vy}-01-01", tz="UTC") - pd.Timedelta(days=7)
    hi = pd.Timestamp(f"{vy + 1}-01-01", tz="UTC") + pd.Timedelta(days=7)
    keep = np.repeat(((train_start <= lo) | (train_start >= hi)).to_numpy(), U * H)
    held = train_year == vy
    dfold = xgb.QuantileDMatrix(X[:n_tr_rows][keep], y_tr[keep], max_bin=128, nthread=N_THREADS)
    booster = train_xgb({**xgb_params, "seed": 0}, dfold, 800)
    oof[held] = booster.predict(xgb.DMatrix(X[:n_tr_rows][np.repeat(held, U * H)], nthread=N_THREADS)).reshape(-1, U, H)
    print(f"out-of-fold {vy} done, {time.time() - t:.0f}s", flush=True)
    del dfold, booster
channel = np.concatenate([oof, p_xgb])     # (n_cases, U, H): out-of-fold on train, final model on test


## 7. Gradient boosting (LightGBM)
The second tree model is LightGBM, trained on the same feature table and the same training rows as XGBoost.
It belongs to the same family (gradient-boosted decision trees) but grows the trees differently: XGBoost grows them level by level up to a maximum depth, while LightGBM always splits the leaf that reduces the error the most, up to a maximum number of leaves. With the same features the two models therefore build different trees and make slightly different errors.

**Hyper-parameters.**

 LightGBM was not tuned with a separate grid search: a single configuration was chosen to be comparable to the validated XGBoost (`num_leaves` 255 for a similar tree capacity, `min_data_in_leaf` 200 as the minimum number of examples implied by XGBoost's `min_child_weight` 50, `feature_fraction` 0.6 and `bagging_fraction` 0.8 as XGBoost's column and row sampling, `lambda_l2` 10 for regularisation, learning rate 0.05).
As for XGBoost, the number of rounds is fixed: early stopping on the three temporal folds stopped at 454, 317 and 459 rounds, and 500 is used for the final model trained on all years.
One seed is used.

**Why it is used together with XGBoost.** 

On its own LightGBM is as good as XGBoost (mean validation AUC 0.98020 vs 0.98030), and the two predictions are very similar.
Averaging them still gives a small but consistent gain, because the part of their errors that differs partly cancels out (variance reduction). Since it costs a single training run, it is kept in the blend with weight 0.30.


In [ ]:
lgb_params = dict(objective="binary", learning_rate=0.05, num_leaves=255, min_data_in_leaf=200,
                  feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=1, lambda_l2=10, max_bin=255,
                  num_threads=N_THREADS, verbose=-1, seed=0)
t = time.time()
model = lgb.train(lgb_params, lgb.Dataset(X[:n_tr_rows], y_tr), 500)
p_lgb = model.predict(X[n_tr_rows:], num_threads=N_THREADS).reshape(-1, U, H)
del model
print(f"lightgbm done, {time.time() - t:.0f}s")


## 8. Sequence neural network
**Idea.** 

The tree models (XGBoost and LightGBM) predict each (case, unit, hour) row independently: to decide whether a unit is on at a given hour they only look at the features of that single row (so only the data corresponding to that specific hour). 
Some of these features already summarise the rest of the week (e.g. rolling means of the margin or weekly price quantiles), but only in the ways we designed by hand; the models never see the other hours of the same unit or the other units directly.

The network instead receives a whole case at once, as a grid of 14 units × 168 hours × all features, and predicts the on/off probability of every cell of the grid together. In this way it can learn by itself how the situation in the other hours of the week and in the other units affects each prediction.

**Inputs.** 

Each feature is standardised (minus its mean, divided by its standard deviation, both computed on the training cases only) so that all features have a similar scale, and extreme values are clipped. Missing values (e.g. features that do not exist for some units) are replaced by 0, with an extra column flagging them.

**Architecture.** 

A case is a grid of 14 units × 168 hours, each cell with its features.
1. Each cell is compressed into a vector of 128 numbers learned by the network, and a learned code identifying the unit is added.
2. Three identical blocks follow, each with:
   - a **bidirectional GRU** along the 168 hours of each unit: it reads the week forwards and backwards, so each
     hour receives information from earlier and later hours;
   - **attention across the 14 units** at each hour: each unit learns how much to look at the others (e.g. units
     of the same plant or sharing the same water);
   - a small feed-forward layer that recombines the information within each cell.
3. A final layer turns the vector of each cell into one probability, giving a 14 × 168 grid of predictions per case.

The size 128 and the number of blocks are hyper-parameters: a first version used 96 and 2 blocks, and the larger network gave a better blend in the temporal cross-validation.


**Training and hyper-parameters.**

 Binary cross-entropy loss, AdamW optimiser (learning rate 1e-3 with a one-cycle schedule, weight decay 1e-3), dropout 0.25, batches of 16 cases, 15 epochs.

These values were chosen with the temporal cross-validation: the validation AUC of the first version peaked between epochs 12 and 18, so training was fixed to 15 epochs with stronger regularisation, and a larger network (size 128, 3 blocks) gave a better blend. The predictions of several seeds are averaged (5 on GPU, 2 on CPU).

**Why it is used.** 

On its own the network is weaker than the tree models, but its errors are quite different from theirs, so adding it to the blend gives the largest gain among all components.


In [ ]:
# Prepares the input of the network: standardises every feature with mean and standard deviation of the
# training rows only, clips extreme values to ±6, replaces missing values with 0 and adds one indicator
# column for each feature that can be missing. Returns a float16 array of shape (case, unit, hour, features).
def to_tensor(X, n_cases, stat_rows):
    """Standardise (stats from training rows only), clip, add NaN indicators -> float16 (n, U, H, F')."""
    nan_cols = np.flatnonzero(np.isnan(X[::97]).any(0))
    S = X[:stat_rows:7]
    mean = np.nanmean(S, 0); std = np.nanstd(S, 0) + 1e-6
    F = X.shape[1]
    Z = np.empty((len(X), F + len(nan_cols)), np.float16)
    for i in range(0, len(X), 500_000):                 # in chunks, to limit memory use
        x = X[i:i + 500_000]; nan = np.isnan(x)
        z = np.clip((x - mean) / std, -6, 6); z[nan] = 0
        Z[i:i + 500_000, :F] = z; Z[i:i + 500_000, F:] = nan[:, nan_cols]
    return Z.reshape(n_cases, U, H, -1)


# The sequence network. Each cell (unit, hour) is projected to a vector of size d and a learned unit code is
# added; then n_layers blocks are applied, each with a bidirectional GRU along the 168 hours of each unit,
# attention across the 14 units at each hour, and a feed-forward layer (each step with a residual connection
# and normalisation). A final linear layer gives one logit per (unit, hour).
class SeqNet(torch.nn.Module):
    def __init__(self, n_feat, d, n_layers, dropout):
        super().__init__()
        nn = torch.nn
        self.inp = nn.Sequential(nn.Linear(n_feat, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(dropout))
        self.unit_emb = nn.Embedding(U, d)
        self.blocks = nn.ModuleList(nn.ModuleDict({
            "gru": nn.GRU(d, d // 2, batch_first=True, bidirectional=True), "n1": nn.LayerNorm(d),
            "att": nn.MultiheadAttention(d, 4, dropout=dropout, batch_first=True), "n2": nn.LayerNorm(d),
            "ff": nn.Sequential(nn.Linear(d, 2 * d), nn.GELU(), nn.Dropout(dropout), nn.Linear(2 * d, d)),
            "n3": nn.LayerNorm(d)}) for _ in range(n_layers))
        self.head = nn.Linear(d, 1)

    def forward(self, x):                                   # x: (B, U, H, F)
        B = x.shape[0]
        h = self.inp(x) + self.unit_emb.weight[None, :, None, :]
        for blk in self.blocks:
            t = h.reshape(B * U, H, -1)                     # along time, per unit
            t = blk["n1"](t + blk["gru"](t)[0])
            a = t.reshape(B, U, H, -1).transpose(1, 2).reshape(B * H, U, -1)   # across units, per hour
            a = blk["n2"](a + blk["att"](a, a, a, need_weights=False)[0])
            a = blk["n3"](a + blk["ff"](a))
            h = a.reshape(B, H, U, -1).transpose(1, 2)
        return self.head(h).squeeze(-1)                     # (B, U, H) logits


# Numerical precision: bf16 mixed precision on GPU (faster, less memory), plain fp32 on CPU.
def autocast():
    # bf16 mixed precision on GPU; plain fp32 on CPU
    return torch.autocast("cuda", dtype=torch.bfloat16) if DEVICE == "cuda" else torch.autocast("cpu", enabled=False)


# Trains one network with a given seed: AdamW optimiser with a one-cycle learning-rate schedule, binary
# cross-entropy loss, batches of bs cases in random order, a fixed number of epochs, gradient clipping.
def train_nn(Ztr, ytr, seed, d, n_layers, epochs=15, bs=16, lr=1e-3, wd=1e-3, dropout=0.25):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = SeqNet(Ztr.shape[-1], d, n_layers, dropout).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    steps = epochs * int(np.ceil(len(Ztr) / bs))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.1)
    lossf = torch.nn.BCEWithLogitsLoss()
    for ep in range(epochs):
        model.train(); perm = rng.permutation(len(Ztr))   # new random order of the cases at every epoch
        for i in range(0, len(perm), bs):
            idx = np.sort(perm[i:i + bs])
            xb = torch.from_numpy(Ztr[idx]).to(DEVICE).float()
            yb = torch.from_numpy(ytr[idx]).to(DEVICE).float()
            with autocast():
                loss = lossf(model(xb).float(), yb)       # error of the predictions on this batch
            opt.zero_grad(); loss.backward()              # compute how each weight should change
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()   # update weights
    return model


# Predicts the given cases in batches, without training (no gradients, dropout off), and converts the
# logits to probabilities with the sigmoid. Returns an array of shape (case, unit, hour).
@torch.no_grad()
def predict_nn(model, Z, bs=32):
    model.eval(); out = []
    for i in range(0, len(Z), bs):
        with autocast():
            out.append(torch.sigmoid(model(torch.from_numpy(Z[i:i + bs]).to(DEVICE).float()).float()).cpu().numpy())
    return np.concatenate(out)


Runs the sequence network. The feature table `X` is converted into the network input `Z` (one 14 × 168 grid
of standardised features per case) and then deleted to free memory, since the tree models are done.
The network is trained on the 2,922 training cases with their labels and predicts the 725 test cases;
this is repeated with `NN_SEEDS` different seeds (5 on GPU, 2 on CPU) and the predictions are averaged
into `p_nn`, of shape (case, unit, hour).


In [ ]:
t = time.time()
Z = to_tensor(X, n_cases, stat_rows=n_tr_rows)
del X                                                    # free ~7.5 GB, the trees are done
print("tensor", Z.shape, f"{Z.nbytes / 1e9:.1f} GB")
p_nn = []
for seed in range(NN_SEEDS):
    model = train_nn(Z[:n_train], labels, seed, d=128, n_layers=3)
    p_nn.append(predict_nn(model, Z[n_train:]))
    print(f"nn seed {seed} done, {time.time() - t:.0f}s", flush=True)
p_nn = np.mean(p_nn, 0)


## 9. Refiner network
The refiner is the same sequence network as in Section 8 (same architecture and hyper-parameters), with one extra input column: the XGBoost probability of each (case, unit, hour) prepared in Section 6, i.e. the out-of-fold prediction on the training cases and `p_xgb` on the test cases.

Before being added, the probability is converted to a logit, log(p / (1 − p)), because probabilities are squeezed close to 0 and 1, while the logit spreads them on a wider scale. It is then standardised with statistics of the training cases only, like all the other inputs. The network can therefore start from the XGBoost estimate and learn, from the whole week and from the other units, where to correct it.

As for the first network, `NN_SEEDS` models with different seeds are trained and their predictions are averaged
into `p_ref`.


In [ ]:
t = time.time()
logit = np.log((channel + 1e-4) / (1 - channel + 1e-4))
logit = (logit - logit[:n_train].mean()) / logit[:n_train].std()      # scaled with training cases only
Z = np.concatenate([Z, logit[..., None].astype(np.float16)], -1)
p_ref = []
for seed in range(NN_SEEDS):
    model = train_nn(Z[:n_train], labels, seed, d=128, n_layers=3)
    p_ref.append(predict_nn(model, Z[n_train:]))
    print(f"refiner seed {seed} done, {time.time() - t:.0f}s", flush=True)
p_ref = np.mean(p_ref, 0)


## 10. Blend and submission
The final prediction is a weighted average, cell by cell, of the four models:

**p = 0.20 · XGBoost + 0.30 · LightGBM + 0.25 · sequence network + 0.25 · refiner**

The weights were chosen by comparing a few combinations with the temporal cross-validation, and confirmed on
2019, a year never used for any other choice. The tree models get 50% in total and the two networks the other
50%: the networks are weaker on their own but make different errors, so mixing the two families gives the
largest gain.

The (case, unit, hour) array is then turned back into the submission format: one row per test case
(`Run No`) and one column per unit and hour (14 × 168 = 2,352 columns), in the same column order as
`sample_submission.csv`. Before writing `submission.csv`, the code checks that the file has 725 rows and
2,353 columns, that all values are probabilities between 0 and 1, and that the cases are in the same order
as `prediction_mapping.csv`.


In [ ]:
p = 0.2 * p_xgb + 0.3 * p_lgb + 0.25 * p_nn + 0.25 * p_ref
sub = pd.DataFrame(p.reshape(len(p), -1), columns=LABEL_COLS)
sub.insert(0, "Run No", cases["Run No"].to_numpy()[n_train:])
sub = sub[pd.read_csv(SAMPLE_CSV, nrows=1).columns]
vals = sub.iloc[:, 1:].to_numpy()
assert sub.shape == (725, 2353) and np.isfinite(vals).all() and vals.min() >= 0 and vals.max() <= 1
assert (sub["Run No"].to_numpy() == mapping["Run No"].to_numpy()).all()
sub.to_csv(OUT_CSV, index=False)
print("wrote", OUT_CSV.resolve(), sub.shape, f"| total runtime {(time.time() - T0) / 3600:.2f} h")
